# Data Collection

This notebook collects raw warning verification data from the IEM Cow (COW) API for 122 NWS Weather Forecast Offices across eleven calendar years (2016–2026).

Data is filtered to three phenomena: Tornado (TO), Severe Thunderstorm (SV), and Flash Flood (FF) warnings. One JSON file is written per WFO-year to `data/01_collection/COW/`. The collection is idempotent; re-running skips files already on disk, so a full re-collection requires clearing the directory first. 2026 is a partial year (collected mid-year) supplying only the Jan 1–Feb 26 2026 tail of the treated window. Project design, data model, and methodological notes are documented in `README.md`.

## Data Source

The data comes from the Iowa Environmental Mesonet (IEM) Cow (COW) tool, a NWS warning verification service maintained by Iowa State University. COW reconstructs the official verification of storm-based warnings: it matches issued warnings against Local Storm Reports (LSRs) using fixed space and time tolerances and computes the standard skill metrics (POD, FAR, CSI, lead time).

Requests are made to the JSON endpoint:

`https://mesonet.agron.iastate.edu/api/1/cow.json`

(documented at https://mesonet.agron.iastate.edu/api/1/docs#/default/cow_handler_api_1_cow_json_get)

Each request fetches one WFO for one full calendar year. The query parameters are:

| Parameter | Value | Purpose |
|---|---|---|
| `wfo` | one office call sign (e.g. `GLD`) | Restricts results to a single Weather Forecast Office |
| `phenomena` | `TO`, `SV`, `FF` (repeated) | Restricts warnings to the three studied phenomena |
| `lsrtype` | `TO`, `SV`, `FF` (repeated) | Restricts storm reports to the matching event types |
| `begints` | `{year}-01-01T00:00Z` | Start of the verification window |
| `endts` | `{year+1}-01-01T00:00Z` | End of the window (exclusive), giving full calendar-year coverage with no boundary overlap |

Multi-valued filters are sent as repeated query parameters (`phenomena=TO&phenomena=SV&...`) rather than comma-joined strings, which is what the IEM API expects.

The matching rules COW applied (LSR buffer, wind and hail thresholds, warning buffer, shared-border setting, and the time window) are echoed back in the `params` block of every response. Because they are recorded per file, the requirement that verification rules be identical across all years is a checkable fact rather than an assumption.

## Implementation

Collection is implemented across three classes in `src/collection/`:

| Class | Responsibility |
|---|---|
| `WFORegistry` | Loads `wfo_list.csv` and provides the 122 WFO call signs used as API query parameters |
| `COWClient` | Wraps the IEM COW API; handles a single WFO-year fetch with retries and exponential backoff |
| `COWCollector` | Orchestrates the full loop across all WFOs and years; skips files already on disk so the run can be safely interrupted and resumed |

Each WFO-year response is written as a separate JSON file to `data/01_collection/COW/{WFO}_{YEAR}.json`. Files are never overwritten because the raw directory is treated as immutable once collected.

In [ ]:
import logging
import sys
from pathlib import Path

sys.path.append("../src")

from collection import COWClient, COWCollector, WFORegistry

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Paths
WFO_FILE = Path("../data/wfo/wfo_list.csv")
COW_DIR  = Path("../data/01_collection/COW")

# IEM COW API endpoint
COW_BASE_URL = "https://mesonet.agron.iastate.edu/api/1/cow.json"

# VTEC phenomena codes to collect and analyze:
#   TO = Tornado Warning
#   SV = Severe Thunderstorm Warning
#   FF = Flash Flood Warning
PHENOMENA = ["TO", "SV", "FF"]

# Analysis window: 2016–2024 are the pre-treatment baseline years; the treatment
# year begins at the staffing-cut anniversary (Feb 27 2025), and the 12-month
# treated window runs through Feb 26 2026 (see README seasonal windowing).
# Whole calendar years are collected so the window can be cut by date downstream.
# 2026 is a PARTIAL year (collected mid-year): it supplies only the Jan 1–Feb 26
# 2026 tail of the treated window and must not be treated as a complete year.
YEARS = list(range(2016, 2027))

# Seconds to pause between API requests to avoid overwhelming the IEM server
RATE_LIMIT = 0.3

In [2]:
registry  = WFORegistry(WFO_FILE)
client    = COWClient(phenomena=PHENOMENA, base_url=COW_BASE_URL)
collector = COWCollector(registry, client, COW_DIR, years=YEARS, rate_limit=RATE_LIMIT)

print(registry)
print(client)
print(collector)

summary = collector.collect()
print(summary)

22:02:05  INFO     Fetching AFC 2016 ...


WFORegistry(122 offices from wfo_list.csv)
COWClient(phenomena=['TO', 'SV', 'FF'], url=https://mesonet.agron.iastate.edu/api/1/cow.json)
COWCollector(122 WFOs, years=2016-2025, output=../data/01_collection/COW)


22:02:06  INFO     Wrote AFC_2016.json
22:02:06  INFO     Fetching AFC 2017 ...
22:02:06  INFO     Wrote AFC_2017.json
22:02:07  INFO     Fetching AFC 2018 ...
22:02:07  INFO     Wrote AFC_2018.json
22:02:07  INFO     Fetching AFC 2019 ...
22:02:08  INFO     Wrote AFC_2019.json
22:02:08  INFO     Fetching AFC 2020 ...
22:02:12  INFO     Wrote AFC_2020.json
22:02:13  INFO     Fetching AFC 2021 ...
22:02:13  INFO     Wrote AFC_2021.json
22:02:13  INFO     Fetching AFC 2022 ...
22:02:13  INFO     Wrote AFC_2022.json
22:02:14  INFO     Fetching AFC 2023 ...
22:02:14  INFO     Wrote AFC_2023.json
22:02:14  INFO     Fetching AFC 2024 ...
22:02:15  INFO     Wrote AFC_2024.json
22:02:15  INFO     Fetching AFC 2025 ...
22:02:15  INFO     Wrote AFC_2025.json
22:02:15  INFO     Fetching AFG 2016 ...
22:02:16  INFO     Wrote AFG_2016.json
22:02:16  INFO     Fetching AFG 2017 ...
22:02:17  INFO     Wrote AFG_2017.json
22:02:17  INFO     Fetching AFG 2018 ...
22:02:17  INFO     Wrote AFG_2018.json
2

{'wrote': 1220, 'skipped': 0, 'failed': 0}
